# 03 · Interface web (Gradio) e API

Interface para o produtor fazer perguntas em linguagem simples. Ela usa **exatamente o pipeline avaliado** no notebook 04 como "Sistema final": busca vetorial → limiar → `Qwen2.5-1.5B-Instruct` + LoRA (4 bits) → JSON validado → resposta com **fonte e link**.

Mudanças em relação à versão anterior:
- o filtro por lista de palavras agrícolas foi removido: ele deixava passar perguntas como "preço do adubo" e bloqueava perguntas válidas sem as palavras da lista. A recusa agora vem do limiar medido e do LLM treinado;
- o adaptador LoRA agora é carregado de fato (antes o modelo era carregado sem ele, em float16);
- cada resposta mostra a fonte oficial e um aviso de segurança;
- corrigido o ícone `alert` que faltava e quebrava a exibição de erros.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/palomari/agrofacil-genai.git"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive")
except ImportError:
    pass  

if not Path("src/agrofacil.py").exists():
    if Path("../src/agrofacil.py").exists():
        os.chdir("..")
    else:
        if not Path("agrofacil-genai").exists():
            subprocess.run(["git", "clone", REPO], check=True)
        os.chdir("agrofacil-genai")

assert Path("src/agrofacil.py").exists(), (
    "Versão antiga do repositório nesta pasta. Apague a pasta agrofacil-genai do Drive e rode de novo.")
sys.path.insert(0, os.getcwd())
print("Raiz do projeto:", os.getcwd())
print("Versão:", subprocess.run(["git", "log", "-1", "--format=%h %cd"], capture_output=True, text=True).stdout.strip())

Mounted at /content/drive
Raiz do projeto: /content/drive/MyDrive/agrofacil-genai
Versão: db82db8 Wed Nov 18 17:44:20 2026 -0300


In [2]:
%pip install -q -r requirements.txt

In [ ]:
"""
Frontend Gradio — AgroFácil GenAI (Envio Direto & Sem Ampulheta).
"""

import html
import inspect
import time
from urllib.parse import quote

import json
import os
import sys

import gradio as gr

sys.path.insert(0, os.getcwd())
from src import agrofacil as af

TITULO = "AgroFácil GenAI"
ICONE_ENVIAR = "→"

SUGESTOES = [
    ("sun", "Saiu aviso de geada para esta noite. Como protejo a lavoura com o pivô?"),
    ("sparkles", "O que significa o aviso laranja do INMET?"),
    ("database", "Como sei a hora de irrigar sem gastar com aparelho?"),
    ("bar", "Como guardar água para a época de seca?"),
    ("list", "O que fazer quando a Defesa Civil emite alerta de granizo?"),
]

ICONES = {
    "sprout": '<path d="M7 20h10"/><path d="M12 20v-8"/><path d="M12 12A5 5 0 0 1 7 7c0-2.5 2-4 5-4 0 2.5 2 4 5 4a5 5 0 0 1-5 5z"/><path d="M12 16a3 3 0 0 0-3-3"/>',
    "sparkles": '<path d="M9.937 15.5A2 2 0 0 0 8.5 14.063l-6.135-1.582a.5.5 0 0 1 0-.962L8.5 9.936A2 2 0 0 0 9.937 8.5l1.582-6.135a.5.5 0 0 1 .963 0L14.063 8.5A2 2 0 0 0 15.5 9.937l6.135 1.581a.5.5 0 0 1 0 .964L15.5 14.063a2 2 0 0 0-1.437 1.437l-1.582 6.135a.5.5 0 0 1-.963 0z"/><path d="M20 3v4"/><path d="M22 5h-4"/><path d="M4 17v2"/><path d="M5 18H3"/>',
    "sun": '<circle cx="12" cy="12" r="4"/><path d="M12 2v2"/><path d="M12 20v2"/><path d="m4.93 4.93 1.41 1.41"/><path d="m17.66 17.66 1.41 1.41"/><path d="M2 12h2"/><path d="M20 12h2"/><path d="m6.34 17.66-1.41 1.41"/><path d="m19.07 4.93-1.41 1.41"/>',
    "moon": '<path d="M12 3a6 6 0 0 0 9 9 9 9 0 1 1-9-9Z"/>',
    "user": '<path d="M19 21v-2a4 4 0 0 0-4-4H9a4 4 0 0 0-4 4v2"/><circle cx="12" cy="7" r="4"/>',
    "database": '<ellipse cx="12" cy="5" rx="9" ry="3"/><path d="M3 5V19A9 3 0 0 0 21 19V5"/><path d="M3 12A9 3 0 0 0 21 12"/>',
    "bar": '<line x1="18" x2="18" y1="20" y2="10"/><line x1="12" x2="12" y1="20" y2="4"/><line x1="6" x2="6" y1="20" y2="14"/>',
    "list": '<line x1="21" x2="3" y1="6" y2="6"/><line x1="15" x2="3" y1="12" y2="12"/><line x1="17" x2="3" y1="18" y2="18"/>',
    "alert": '<circle cx="12" cy="12" r="10"/><line x1="12" x2="12" y1="8" y2="12"/><line x1="12" x2="12.01" y1="16" y2="16"/>',
}

def svg(nome, tam=18, classe=""):
    return (
        f'<svg class="{classe}" xmlns="http://www.w3.org/2000/svg" width="{tam}" height="{tam}" '
        'viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" '
        f'stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">{ICONES[nome]}</svg>'
    )

def uri_svg(nome, cor="#000000"):
    desenho = (
        "<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 24 24' fill='none' "
        f"stroke='{cor}' stroke-width='2' stroke-linecap='round' stroke-linejoin='round'>"
        f"{ICONES[nome].replace(chr(34), chr(39))}</svg>"
    )
    return "data:image/svg+xml," + quote(desenho, safe="")

TEMA_ONCLICK = (
    "event.stopPropagation();document.body.classList.toggle('dark');"
    "try{localStorage.setItem('tema',document.body.classList.contains('dark')?'escuro':'claro')}catch(e){}"
)

JS_CARGA = """
() => {
  if (window.__agroPronto) return;
  window.__agroPronto = true;
  try {
    document.body.classList.toggle('dark', localStorage.getItem('tema') === 'escuro');
  } catch (e) {
    document.body.classList.remove('dark');
  }
  document.addEventListener('click', (ev) => {
    const alvo = ev.target.closest('[data-acao]');
    if (!alvo) return;
    if (alvo.dataset.acao === 'tema') {
      const escuro = document.body.classList.toggle('dark');
      try { localStorage.setItem('tema', escuro ? 'escuro' : 'claro'); } catch (e) {}
    } else if (alvo.dataset.acao === 'topo') {
      window.location.reload();
    }
  });
}
"""

JS_ROLAR = """
() => {
  const msgs = document.querySelectorAll('#chat-history .msg.user');
  if (msgs.length) msgs[msgs.length - 1].scrollIntoView({behavior: 'smooth', block: 'start'});
}
"""

def html_header():
    return (
        '<div class="header-inner">'
        '<div class="brand" data-acao="topo" title="Voltar ao início" style="cursor: pointer;">'
        f'<div class="brand-icon">{svg("sprout", 22)}</div>'
        f"<span>{TITULO}</span></div>"
        '<button type="button" class="theme-toggle" data-acao="tema" '
        'title="Alternar tema claro/escuro" aria-label="Alternar tema claro/escuro" '
        f'onclick="{TEMA_ONCLICK}"><span class="i-sun">{svg("sun", 19)}</span><span class="i-moon">{svg("moon", 19)}</span></button>'
        "</div>"
    )

INTRO_HTML = """
<div class="welcome">
    <div class="greeting">🌱 Assistente de segurança climática para o produtor</div>
    <h1>Geada, seca, chuva forte: o que fazer?</h1>
    <p>Pergunte em linguagem simples. As respostas vêm só de fontes oficiais<br>
    (INMET, Defesa Civil, Embrapa, EPAMIG) e mostram de onde vieram.</p>
</div>
"""

def html_usuario(pergunta):
    return (
        '<div class="msg user"><div class="bubble">' + html.escape(pergunta) + "</div>"
        f'<div class="av av-user">{svg("user", 19)}</div></div>'
    )

def html_bot(conteudo):
    return (
        f'<div class="msg bot"><div class="av av-bot">{svg("sparkles", 18)}</div>'
        f'<div class="answer">{conteudo}</div></div>'
    )

def html_pensando():
    return html_bot(
        '<div class="typing"><span></span><span></span><span></span>'
        "<em>Gerando resposta técnica…</em></div>"
    )

def status(texto, erro=False):
    icone = "alert" if erro else "sparkles"
    classe = "status erro" if erro else "status"
    return f'<div class="{classe}">{svg(icone, 16)}<span>{html.escape(texto)}</span></div>'

def html_resposta(r, erro=None):
    """Recebe o dicionário devolvido por af.responder e mostra resposta, fonte e aviso."""
    if erro is not None:
        return html_bot(status(f"Erro ao gerar resposta: {erro}", erro=True))
    texto = r["resposta"] if isinstance(r, dict) else str(r)
    corpo = f'<div class="text-response" style="line-height: 1.6; font-size: 14.5px;">{html.escape(texto)}</div>'
    if isinstance(r, dict) and r.get("status") == "ok" and r.get("fonte"):
        corpo += (
            '<div class="fonte" style="margin-top:10px;font-size:12.5px;color:var(--muted);">'
            f'Fonte [{html.escape(str(r["fonte_id"]))}]: '
            f'<a href="{html.escape(r.get("url") or "")}" target="_blank" rel="noopener">{html.escape(r["fonte"])}</a></div>'
        )
    if isinstance(r, dict) and r.get("aviso"):
        corpo += f'<div style="margin-top:8px;font-size:12px;color:var(--muted);">{html.escape(r["aviso"])}</div>'
    return html_bot(corpo)

def html_falha(exc):
    return html_bot(status(f"Erro inesperado: {exc}", erro=True))

CSS_BASE = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

:root {
    --text: #1b3a2b;
    --text-soft: #2d5a43;
    --muted: #6b8c7a;
    --border: #d4e5dc;
    --accent: #2e8b57;
    --accent-soft: #e8f5ed;
    --card: linear-gradient(180deg, #f3faf5 0%, #ebf5ef 100%);
    --page: linear-gradient(135deg, #e3f2e8 0%, #e8f5ed 55%, #def0e4 100%);
    --surface: #ffffff;
    --surface-2: #f4fcf6;
    --bubble: #d2ebd9;
    --bubble-text: #133322;
    --ok-bg: #eefaf3;
    --ok-text: #23915f;
    --err-bg: #fff0f0;
    --err-text: #c0392b;
    --shadow: 0 10px 34px rgba(46, 139, 87, .08);
}

body.dark {
    --text: #e2f4ea;
    --text-soft: #b8dcc7;
    --muted: #85b098;
    --border: #2a4738;
    --accent: #4ade80;
    --accent-soft: #1e382b;
    --card: linear-gradient(180deg, #15261d 0%, #112018 100%);
    --page: linear-gradient(135deg, #0b1410 0%, #0f1c15 100%);
    --surface: #1a2e23;
    --surface-2: #15261d;
    --bubble: #254a36;
    --bubble-text: #e2f4ea;
    --ok-bg: #16322a;
    --ok-text: #4cd29a;
    --err-bg: #fff0f0;
    --err-text: #ff8d8d;
    --shadow: 0 10px 34px rgba(0, 0, 0, .35);
}

body, gradio-app {
    background: var(--page) fixed !important;
    min-height: 100vh;
}

.gradio-container {
    --body-background-fill: transparent;
    --block-border-width: 0px;
    --block-shadow: none;
    font-family: 'Inter', system-ui, -apple-system, sans-serif !important;
    width: calc(100% - 48px) !important;
    max-width: 1180px !important;
    margin: 24px auto !important;
    padding: 0 !important;
    background: var(--card) !important;
    border: 1px solid var(--border) !important;
    border-radius: 24px !important;
    box-shadow: var(--shadow);
    color: var(--text);
}

footer, .built-with, .show-api { display: none !important; }

#app-header, #chat-history, #welcome-area .block, #welcome-area .prose {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    padding: 0 !important;
}

#app-header .header-inner {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 26px 44px;
}

.brand {
    display: flex;
    align-items: center;
    gap: 14px;
    color: var(--text);
    font-size: 18px;
    font-weight: 600;
}

.brand-icon {
    width: 42px;
    height: 42px;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    border-radius: 13px;
    background: var(--accent-soft);
    color: var(--accent);
}

.theme-toggle {
    width: 42px;
    height: 42px;
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    border-radius: 50%;
    border: 1px solid var(--border);
    background: var(--surface);
    color: var(--accent);
    cursor: pointer;
    padding: 0 !important;
}

.theme-toggle .i-moon { display: none !important; }
.theme-toggle .i-sun { display: inline-block !important; }
body.dark .theme-toggle .i-sun { display: none !important; }
body.dark .theme-toggle .i-moon { display: inline-block !important; stroke: #ffffff !important; }

.welcome {
    text-align: center;
    margin: 34px auto 34px;
    max-width: 700px;
    padding: 0 20px;
}

.welcome .greeting {
    color: var(--muted);
    font-size: 15px;
    margin-bottom: 12px;
}

.welcome h1 {
    color: var(--text) !important;
    font-size: clamp(24px, 3vw, 31px) !important;
    font-weight: 700 !important;
    margin: 0 0 12px !important;
}

.welcome p {
    color: var(--muted) !important;
    font-size: 14px !important;
    line-height: 1.65 !important;
}

#suggestions {
    display: grid !important;
    grid-template-columns: repeat(5, minmax(0, 1fr));
    gap: 14px !important;
    max-width: 980px;
    width: calc(100% - 40px);
    margin: 0 auto 30px !important;
    border: none !important;
    background: transparent !important;
}

.sug {
    display: flex !important;
    flex-direction: column !important;
    align-items: flex-start !important;
    justify-content: space-between !important;
    gap: 16px;
    min-height: 110px;
    padding: 16px !important;
    text-align: left !important;
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: 16px !important;
    color: var(--text-soft) !important;
    font-size: 12.5px !important;
    font-weight: 500 !important;
    line-height: 1.45 !important;
    box-shadow: 0 6px 22px rgba(46, 139, 87, .05) !important;
    transition: transform .15s ease, border-color .15s ease !important;
}

.sug:hover {
    transform: translateY(-2px);
    border-color: var(--accent) !important;
}

.sug::before {
    content: "";
    width: 20px;
    height: 20px;
    background-color: var(--text-soft);
    -webkit-mask-repeat: no-repeat;
    mask-repeat: no-repeat;
    -webkit-mask-position: center;
    mask-position: center;
    -webkit-mask-size: contain;
    mask-size: contain;
}

#question-bar {
    display: flex !important;
    align-items: center !important;
    gap: 12px !important;
    max-width: 720px;
    width: calc(100% - 40px);
    margin: 0 auto 28px !important;
    padding: 10px 12px 10px 16px !important;
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: 999px !important;
    box-shadow: 0 10px 30px rgba(46, 139, 87, .09);
}

#question-input textarea {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
    resize: none !important;
    font-size: 15px !important;
    color: var(--text) !important;
}

#send-btn {
    flex: 0 0 auto !important;
    width: 64px !important;
    height: 40px !important;
    border-radius: 20px !important;
    border: none !important;
    background: var(--accent-soft) !important;
    color: var(--accent) !important;
    font-size: 22px !important;
    font-weight: 600 !important;
}

#chat-history {
    max-width: 800px;
    width: 100%;
    margin: 0 auto;
    padding: 0 28px !important;
}

.msg {
    display: flex;
    align-items: flex-start;
    gap: 12px;
    margin: 22px 0;
}

.msg.user { justify-content: flex-end; }

.bubble {
    max-width: 78%;
    padding: 13px 20px;
    border-radius: 18px;
    background: var(--bubble);
    color: var(--bubble-text);
    font-size: 15px;
    line-height: 1.5;
}

.av-user {
    width: 38px;
    height: 38px;
    border-radius: 50%;
    background: var(--accent-soft);
    color: var(--accent);
    display: flex;
    align-items: center;
    justify-content: center;
}

.av-bot {
    width: 34px;
    height: 34px;
    border-radius: 11px;
    background: linear-gradient(135deg, #2e8b57, #3cb371);
    color: #ffffff;
    display: flex;
    align-items: center;
    justify-content: center;
    box-shadow: 0 6px 16px rgba(46, 139, 87, .3);
}

.answer {
    flex: 1 1 auto;
    max-width: calc(100% - 50px);
    padding: 16px;
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 16px;
    box-shadow: var(--shadow);
    color: var(--text);
}

.status {
    display: flex;
    align-items: center;
    gap: 8px;
    padding: 10px 14px;
    border-radius: 10px;
    background: var(--ok-bg);
    color: var(--ok-text);
    font-size: 12.5px;
}

.typing {
    display: flex;
    align-items: center;
    gap: 5px;
    color: var(--muted);
    font-size: 13px;
}
.typing span {
    width: 7px; height: 7px; border-radius: 50%; background: var(--accent);
    animation: pulsar 1.2s infinite ease-in-out;
}
.typing span:nth-child(2) { animation-delay: .15s; }
.typing span:nth-child(3) { animation-delay: .3s; }
@keyframes pulsar { 0%, 80%, 100% { opacity: .25; transform: scale(.8); } 40% { opacity: 1; transform: scale(1); } }
"""

def css_icones():
    regras = []
    for i, (icone, _) in enumerate(SUGESTOES, 1):
        uri = uri_svg(icone)
        regras.append(f'.sug-{i}::before {{ -webkit-mask-image: url("{uri}"); mask-image: url("{uri}"); }}')
    return "\n".join(regras)

CSS = CSS_BASE + css_icones()

def _aceita(func, nome):
    try:
        return nome in inspect.signature(func).parameters
    except (TypeError, ValueError):
        return False

def _ligar(gatilho, fn, entradas, saidas):
    try:
        evento = gatilho(fn, inputs=entradas, outputs=saidas, show_progress="hidden")
    except TypeError:
        evento = gatilho(fn, inputs=entradas, outputs=saidas)
    try:
        evento.then(None, None, None, js=JS_ROLAR)
    except Exception:
        pass

def construir_interface(gerar):
    def responder(texto_pergunta, historico_atual):
        historico_atual = historico_atual or ""
        pergunta = str(texto_pergunta or "").strip()
        if not pergunta:
            yield historico_atual, "", gr.update(value=ICONE_ENVIAR, interactive=True), gr.update()
            return

        esconder = gr.update(visible=False)
        usuario = html_usuario(pergunta)

        yield (historico_atual + usuario + html_pensando(), "",
               gr.update(value=ICONE_ENVIAR, interactive=False), esconder)

        try:
            resposta_texto = gerar(pergunta, temperature=0.7, top_p=0.9, max_tokens=128)
            resposta = html_resposta(resposta_texto)
        except Exception as exc:
            resposta = html_falha(exc)

        yield (historico_atual + usuario + resposta, "",
               gr.update(value=ICONE_ENVIAR, interactive=True), esconder)

    tema = gr.themes.Soft(primary_hue="emerald", secondary_hue="green", neutral_hue="slate", radius_size="lg")
    blocks_kwargs = {"title": TITULO}
    launch_kwargs = {}
    for nome, valor in (("theme", tema), ("css", CSS), ("js", JS_CARGA)):
        if _aceita(gr.Blocks.__init__, nome):
            blocks_kwargs[nome] = valor
        elif _aceita(gr.Blocks.launch, nome):
            launch_kwargs[nome] = valor

    with gr.Blocks(**blocks_kwargs) as demo:
        gr.HTML(html_header(), elem_id="app-header")

        with gr.Column(elem_id="welcome-area") as boas_vindas:
            gr.HTML(INTRO_HTML)
            with gr.Row(elem_id="suggestions"):
                botoes = [
                    gr.Button(texto, elem_classes=["sug", f"sug-{i}"])
                    for i, (_, texto) in enumerate(SUGESTOES, 1)
                ]

        historico = gr.HTML(value="", elem_id="chat-history")

        with gr.Row(elem_id="question-bar"):
            pergunta = gr.Textbox(
                placeholder="Faça sua pergunta sobre a lavoura aqui...",
                lines=1,
                max_lines=4,
                show_label=False,
                container=False,
                elem_id="question-input",
            )
            btn = gr.Button(ICONE_ENVIAR, variant="primary", elem_id="send-btn")

        entradas = [pergunta, historico]
        saidas = [historico, pergunta, btn, boas_vindas]

        _ligar(btn.click, responder, entradas, saidas)
        _ligar(pergunta.submit, responder, entradas, saidas)

        for botao, (_, texto) in zip(botoes, SUGESTOES):
            _ligar(
                botao.click,
                responder,
                [gr.State(texto), historico],
                saidas,
            )

    return demo, launch_kwargs

def gerador_demo():
    def gerar(pergunta, temperature, top_p, max_tokens):
        time.sleep(0.5)
        return {"resposta": f"(modo demonstração, sem modelo) Pergunta recebida: {pergunta}",
                "status": "demo", "fonte": None, "aviso": af.AVISO_SEGURANCA}
    return gerar

def main():
    DEMO = False  
    estado = {}

    def gerar(pergunta, temperature=None, top_p=None, max_tokens=None):
        return estado["fn"](pergunta, temperature, top_p, max_tokens)

    demo, launch_kwargs = construir_interface(gerar)

    if DEMO:
        estado["fn"] = gerador_demo()
    else:
        cfg = af.carregar_config()
        print("Configuração (notebook 01):", cfg)
        buscador = af.Buscador(af.carregar_base(), cfg["encoder"])
        adapter = af.CAMINHO_ADAPTER if af.CAMINHO_ADAPTER.exists() else None
        if adapter is None:
            print("AVISO: adaptador LoRA não encontrado em modelos/agrofacil_adapter (rode o notebook 02). Usando o modelo base.")
        model, tok = af.carregar_modelo(adapter=adapter)
        print("Modelo carregado em 4 bits | LoRA:", adapter is not None)

        def _gerar_real(pergunta, *_):
            r = af.responder(pergunta, model, tok, buscador=buscador, modo="rag",
                             limiar=cfg["limiar"], k=cfg.get("k", 3))
            print(json.dumps({"pergunta": af.mascarar_pii(pergunta), "status": r["status"],
                              "fonte_id": r["fonte_id"], "score_top1": r["score_top1"]}, ensure_ascii=False))
            return r

        estado["fn"] = _gerar_real

    demo.launch(share=True, **launch_kwargs)


if __name__ == "__main__":
    main()

Configuração (notebook 01): {'encoder': 'sentence-transformers/all-MiniLM-L6-v2', 'limiar': 0.425, 'k': 3}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado em 4 bits | LoRA: True
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb9d4909b0a1bf37b7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## API REST (opcional)

A mesma lógica também é exposta como API em `api/app.py` (FastAPI). Ela carrega uma segunda cópia do modelo; em 4 bits as duas cabem na T4.

In [ ]:
import subprocess, time, requests
api = subprocess.Popen(["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8000"],
                       stdout=open("api.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90): 
    try:
        if requests.get("http://localhost:8000/saude", timeout=2).ok:
            break
    except Exception:
        time.sleep(5)
print(requests.get("http://localhost:8000/saude").json())
r = requests.post("http://localhost:8000/perguntar",
                  json={"pergunta": "Como faço para receber os alertas da Defesa Civil no celular?"})
print(r.status_code, r.json())

{"pergunta": "O que significa o aviso laranja do INMET?", "status": "ok", "fonte_id": "T02", "score_top1": 0.5490323305130005}
{'status': 'ok', 'modelo': 'Qwen/Qwen2.5-1.5B-Instruct', 'lora': True, 'config': {'encoder': 'sentence-transformers/all-MiniLM-L6-v2', 'limiar': 0.425, 'k': 3}}
200 {'resposta': 'Envie um SMS com o CEP para o número 40199, ou fale com a Defesa Civil pelo WhatsApp (61) 2034-4611. Também tem o sistema Defesa Civil Alerta que envia mensagens direto para os celulares da área atingida.', 'status': 'ok', 'fonte_id': 'T03', 'fonte': 'INMET - Aviso meteorológico ou alerta? Entenda a diferença', 'url': 'https://portal.inmet.gov.br/noticias/aviso-meteorológico-e-alerta-entenda-a-diferença', 'score_top1': 0.7269244194030762, 'trechos_recuperados': [{'id': 'T03', 'score': 0.7269}, {'id': 'T21', 'score': 0.6185}, {'id': 'T02', 'score': 0.5889}], 'aviso': 'Orientação informativa baseada em fontes oficiais. Não substitui a Defesa Civil nem a assistência técnica. Em emergência

## Conclusão

- **Interface e UX:** tema visual da equipe mantido, sugestões de perguntas sobre geada, seca e alertas, e resposta com fonte clicável.
- **Integração:** a interface e a API importam `src/agrofacil.py`, o mesmo código medido no notebook 04, sem lógica duplicada.
- **Recusa segura:** perguntas fora da base são recusadas pelo limiar ou pelo LLM com *"Não consta na base de conhecimento."*.
- **LGPD:** nenhuma pergunta é armazenada; o log de operação mascara CPF, telefone e e-mail.